In [1]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.datasets as datasets
from torchvision import models
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from torch.utils.data.distributed import DistributedSampler
from tqdm import tqdm
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
from torch_xla import runtime as xr
#import torchmetrics
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="jax")
os.environ.pop('TPU_PROCESS_ADDRESSES')

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


'local'

In [2]:
class FilteredImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes, _ = super().find_classes(directory)
        excluded_folder = 'n02109961-Eskimo_dog'
        
        classes = [c for c in classes if c != excluded_folder]
        classes.sort()
        
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        
        return classes, class_to_idx

In [3]:
def merge_classes(dataset):
    eskimo_folder = 'n02109961-Eskimo_dog'
    husky_folder = 'n02110185-Siberian_husky'
    
    old_class_to_idx = dataset.class_to_idx
    new_class_to_idx = {}
    remap_dict = {}
    next_idx = 0
    
    for class_name, old_idx in sorted(old_class_to_idx.items(), key=lambda x: x[1]):
        if class_name == eskimo_folder:
            continue
        new_class_to_idx[class_name] = next_idx
        remap_dict[old_idx] = next_idx
        next_idx += 1
        
    remap_dict[old_class_to_idx[eskimo_folder]] = new_class_to_idx[husky_folder]
    new_samples = [(path, remap_dict[old_idx]) for path, old_idx in dataset.samples]
    
    dataset.samples = new_samples
    dataset.imgs = new_samples
    dataset.class_to_idx = new_class_to_idx
    dataset.classes = list(new_class_to_idx.keys())
    return dataset

In [4]:
path = '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images/'

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

"""
validation_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
"""

validation_transforms=transforms.Compose([
    transforms.Resize(236, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_full = merge_classes(datasets.ImageFolder(root=path, transform=train_transforms))
val_full = merge_classes(datasets.ImageFolder(root=path, transform=validation_transforms))

torch.manual_seed(42)
train_size = int(len(train_full) * 0.8)
indices = torch.randperm(len(train_full)).tolist()

print(f"Training: {train_size}, Validation: {int(len(val_full) * 0.2)}")

training = Subset(train_full, indices[:train_size])
validation = Subset(val_full, indices[train_size:])

Training: 16464, Validation: 4116


In [5]:
def train_model(model, train_loader, train_sampler, val_loader, criterion, optimizer, scheduler, num_epochs):
    is_master = xm.is_master_ordinal()
    for epoch in range(num_epochs):
        model.train()
        train_sampler.set_epoch(epoch)
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", unit="batch", disable=not is_master)
        for inputs, labels in train_pbar:
            #inputs, labels = inputs.to(device), labels.to(device)
                    
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            xm.optimizer_step(optimizer)
        
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels.data).item()
            train_total += inputs.size(0)
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", unit="batch", leave=True, disable=not is_master)
            for inputs, labels in val_pbar:
                #inputs, labels = inputs.to(device), labels.to(device)
                    
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                        
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data).item()
                val_total += inputs.size(0)

        """
        train_correct = xm.mesh_reduce('train_correct', torch.tensor(train_correct), sum)
        train_total = xm.mesh_reduce('train_total', torch.tensor(train_total), sum)
        val_correct = xm.mesh_reduce('val_correct', torch.tensor(val_correct), sum)
        val_total = xm.mesh_reduce('val_total', torch.tensor(val_total), sum)

        train_loss = xm.mesh_reduce('train_loss', torch.tensor(train_loss), sum)
        val_loss = xm.mesh_reduce('val_loss', torch.tensor(val_loss), sum)
        avg_train_loss = train_loss.item() / train_total
        avg_val_loss = val_loss.item() / val_total

        t_acc = 100 * train_correct.double() / train_total
        v_acc = 100 * val_correct.double() / val_total
        """

        metrics = torch.tensor(
            [train_correct, train_total, val_correct, val_total, train_loss, val_loss],
            dtype=torch.float64,
            device=torch_xla.device()
        )

        synced_metrics = xm.all_reduce(xm.REDUCE_SUM, metrics)
        tc, tt, vc, vt, tl, vl = synced_metrics.tolist()

        avg_train_loss = tl / tt
        avg_val_loss = vl / vt
        t_acc = 100 * tc / tt
        v_acc = 100 * vc / vt
        
        scheduler.step(v_acc)
        
        if is_master:
            print(f"Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")
            print(f"Training Accuracy: {t_acc:.2f}%, Validation Accuracy: {v_acc:.2f}%")

In [6]:
def fine_tune(model, train_loader, train_sampler, val_loader, criterion, num_epochs, lr):
    for parameter in model.parameters():
        parameter.requires_grad = True
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=4)
    if xm.is_master_ordinal():
        print('\nFine Tuning:\n')
    train_model(model, train_loader, train_sampler, val_loader, criterion, optimizer, scheduler, num_epochs)

In [7]:
def _mp_fn(index):
    device = torch_xla.device()
    lr1 = 25e-5 * xr.world_size()
    lr2 = 2.5e-6 * xr.world_size()
    
    train_sampler = DistributedSampler(training, num_replicas=xr.world_size(), rank=xr.global_ordinal(), shuffle=True)
    validation_sampler = DistributedSampler(validation, num_replicas=xr.world_size(), rank=xr.global_ordinal(), shuffle=False)
    
    train_loader = DataLoader(training, batch_size=32, sampler=train_sampler, num_workers=12, shuffle=False)
    validation_loader = DataLoader(validation, batch_size=32, sampler=validation_sampler, num_workers=12, shuffle=False)
    
    train_loader_xla = pl.MpDeviceLoader(train_loader, device)
    validation_loader_xla = pl.MpDeviceLoader(validation_loader, device)
    #model = models.convnext_base(weights='DEFAULT')
    model = models.convnext_small(weights='DEFAULT')
    for parameter in model.parameters():
        parameter.requires_grad = False
    
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, 119)
    model.to(device)
    xm.broadcast_master_param(model)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    optimizer = optim.Adam(model.classifier[2].parameters(), lr=lr1)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=4)
    #scheduler2 = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    train_model(model, train_loader_xla, train_sampler, validation_loader_xla, criterion, optimizer, scheduler, num_epochs=20)
    fine_tune(model, train_loader_xla, train_sampler, validation_loader_xla, criterion, num_epochs=10, lr=lr2)

    if xm.is_master_ordinal():
        print("Saving model")

    xm.save(model.state_dict(), 'dog-classifier-ConvNextSmall-webapp_merged-classes.pth')

In [8]:
torch_xla.launch(_mp_fn, args=(), start_method='fork')

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


  0%|          | 0.00/192M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


 41%|████      | 78.8M/192M [00:00<00:01, 93.4MB/s]

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


...

Training Loss: 1.1414, Validation Loss: 0.9558
Training Accuracy: 90.15%, Validation Accuracy: 96.21%


Epoch 9 [Val]: 100%|██████████| 17/17 [00:05<00:00,  2.91batch/s]


Training Loss: 1.1394, Validation Loss: 0.9585
Training Accuracy: 90.14%, Validation Accuracy: 96.21%


Epoch 10 [Val]: 100%|██████████| 17/17 [00:05<00:00,  2.88batch/s]


Training Loss: 1.1611, Validation Loss: 0.9552
Training Accuracy: 89.64%, Validation Accuracy: 96.14%
Saving model
